# 🛰️ Route Resilience AI — SegFormer Road Extraction
## Google Colab Training Notebook

**Model**: SegFormer-B2 (nvidia/mit-b2) fine-tuned for binary road segmentation  
**Dataset**: SpaceNet Road Detection v2 (Vegas, Paris, Shanghai, Khartoum)  
**Loss**: 0.5 × Dice + 0.5 × BCE  
**Target Inference**: Coimbatore, Tamil Nadu satellite imagery  

---
### Setup checklist before running:
1. ✅ Runtime → Change runtime type → **GPU (T4 or A100)**
2. ✅ Mount Google Drive (cell below)
3. ✅ AWS credentials configured
4. ✅ Google Earth Engine authenticated


In [ ]:
# ── Step 1: Mount Google Drive ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/reroute-ai'
os.makedirs(f'{DRIVE_BASE}/weights', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/coimbatore_data', exist_ok=True)
print('✅ Drive mounted. Base:', DRIVE_BASE)

In [ ]:
# ── Step 2: Install Dependencies ─────────────────────────────────────────
!pip install -q \
    transformers==4.41.1 \
    accelerate==0.30.0 \
    albumentations==1.4.8 \
    earthengine-api==0.1.405 \
    rasterio==1.3.10 \
    boto3==1.34.115 \
    awscli==1.32.115 \
    scikit-image==0.23.2 \
    loguru==0.7.2 \
    tqdm==4.66.4 \
    matplotlib==3.9.0 \
    wandb==0.17.0

print('✅ Dependencies installed')

In [ ]:
# ── Step 3: Verify GPU ───────────────────────────────────────────────────
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU found. Training will be very slow. Enable GPU in Runtime settings.')

In [ ]:
# ── Step 4: Configuration ────────────────────────────────────────────────
import os

CFG = {
    # Coimbatore AOI [xmin, ymin, xmax, ymax]
    'coimbatore_bbox': [76.85, 10.90, 77.10, 11.15],
    'coimbatore_center': (11.0168, 76.9558),

    # Model
    'model_id': 'nvidia/mit-b2',
    'image_size': 512,
    'num_labels': 1,

    # Training
    'batch_size': 8,
    'num_epochs': 50,
    'lr': 6e-5,
    'weight_decay': 0.01,
    'warmup_epochs': 3,
    'dice_weight': 0.5,
    'grad_clip': 1.0,

    # Data
    'dataset_dir': '/content/spacenet',
    'train_split': 0.8,
    'val_split': 0.2,

    # Checkpoints
    'weights_dir': f'{DRIVE_BASE}/weights',
    'best_model_name': 'best_model.pth',
    'checkpoint_every': 5,

    # SpaceNet
    'spacenet_cities': ['Vegas'],   # Start with Vegas (largest, most roads)
}

print('Config loaded:')
for k, v in CFG.items():
    print(f'  {k}: {v}')

In [ ]:
# ── Step 5: Configure AWS Credentials ───────────────────────────────────
# Option A: Paste credentials directly (for Colab session only, not saved)
import os

# Fill in your credentials:
AWS_ACCESS_KEY = ''      # paste your key
AWS_SECRET_KEY = ''      # paste your secret
AWS_REGION    = 'us-east-1'

if AWS_ACCESS_KEY and AWS_SECRET_KEY:
    os.environ['AWS_ACCESS_KEY_ID'] = AWS_ACCESS_KEY
    os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET_KEY
    os.environ['AWS_DEFAULT_REGION'] = AWS_REGION
    print('✅ AWS credentials set from variables')
else:
    print('⚠️  No credentials set — using AWS CLI config if available')
    !aws sts get-caller-identity

In [ ]:
# ── Step 6: Download SpaceNet Road Detection Dataset ─────────────────────
import subprocess, os

os.makedirs(CFG['dataset_dir'], exist_ok=True)

SPACENET_CITY_PREFIXES = {
    'Vegas':    'AOI_2_Vegas',
    'Paris':    'AOI_3_Paris',
    'Shanghai': 'AOI_4_Shanghai',
    'Khartoum': 'AOI_5_Khartoum',
}

for city in CFG['spacenet_cities']:
    prefix = SPACENET_CITY_PREFIXES[city]
    s3_path = f's3://spacenet-dataset/spacenet/SN3_roads/train/{prefix}/'
    local_path = f"{CFG['dataset_dir']}/{prefix}"
    print(f'Downloading {city} from {s3_path} ...')
    !aws s3 sync {s3_path} {local_path} --request-payer requester --quiet
    print(f'✅ {city} downloaded to {local_path}')

# Show structure
!find {CFG['dataset_dir']} -maxdepth 3 -type d | head -30

In [ ]:
# ── Step 7: Explore Dataset ───────────────────────────────────────────────
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from PIL import Image

# Find image and mask files
image_files = sorted(glob.glob(f"{CFG['dataset_dir']}/**/RGB-PanSharpen/*.tif", recursive=True))
mask_files  = sorted(glob.glob(f"{CFG['dataset_dir']}/**/geojson_roads_speed/**/*.geojson", recursive=True))

print(f'Found {len(image_files)} satellite image tiles')
print(f'Found {len(mask_files)} road annotation files')

# Visualise first tile
if image_files:
    with rasterio.open(image_files[0]) as src:
        img = src.read([1, 2, 3]).transpose(1, 2, 0)
        img = (img / img.max() * 255).astype(np.uint8)

    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(img)
    ax.set_title(f'SpaceNet tile: {os.path.basename(image_files[0])}')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Image shape: {img.shape}')

In [ ]:
# ── Step 8: Preprocessing — Generate Image/Mask Chip Pairs ───────────────
import os, glob
import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from shapely.geometry import shape
import json
from PIL import Image
from tqdm import tqdm

CHIP_DIR = '/content/chips'
IMG_DIR  = f'{CHIP_DIR}/images'
MASK_DIR = f'{CHIP_DIR}/masks'
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)

def process_tile(img_path: str, geojson_dir: str, chip_size: int = 512):
    """
    Read one SpaceNet image tile and its road GeoJSON, rasterize roads to binary mask.
    Save as PNG pair.
    """
    tile_id = os.path.basename(img_path).replace('.tif', '')

    # Find matching geojson
    geojson_paths = glob.glob(f'{geojson_dir}/**/*{tile_id}*.geojson', recursive=True)
    if not geojson_paths:
        return None

    with rasterio.open(img_path) as src:
        img = src.read([1, 2, 3]).transpose(1, 2, 0)  # H, W, 3
        transform = src.transform
        crs = src.crs
        h, w = img.shape[:2]

        # Normalise to uint8
        img_norm = np.clip(img / 2000.0 * 255, 0, 255).astype(np.uint8)

        # Rasterize road geometries
        with open(geojson_paths[0]) as f:
            roads = json.load(f)

        geometries = [
            shape(feat['geometry'])
            for feat in roads.get('features', [])
            if feat['geometry'] is not None
        ]

        if geometries:
            mask = rasterize(
                [(geom, 1) for geom in geometries],
                out_shape=(h, w),
                transform=transform,
                fill=0,
                dtype=np.uint8,
            )
            # Dilate road mask slightly (roads are 1–2 px wide)
            from skimage.morphology import dilation, disk
            mask = dilation(mask, disk(2))
        else:
            mask = np.zeros((h, w), dtype=np.uint8)

    # Save
    Image.fromarray(img_norm).save(f'{IMG_DIR}/{tile_id}.png')
    Image.fromarray(mask * 255).save(f'{MASK_DIR}/{tile_id}.png')
    return tile_id


image_files = sorted(glob.glob(f"{CFG['dataset_dir']}/**/RGB-PanSharpen/*.tif", recursive=True))
geojson_dir = CFG['dataset_dir']

processed = 0
for img_path in tqdm(image_files, desc='Processing tiles'):
    result = process_tile(img_path, geojson_dir)
    if result:
        processed += 1

print(f'\n✅ Processed {processed} chip pairs → {CHIP_DIR}')
print(f'  Images: {len(os.listdir(IMG_DIR))}')
print(f'  Masks:  {len(os.listdir(MASK_DIR))}')

In [ ]:
# ── Step 9: Visualise Image/Mask Pairs ───────────────────────────────────
import random, os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

img_files  = sorted(os.listdir(IMG_DIR))
mask_files = sorted(os.listdir(MASK_DIR))

samples = random.sample(range(min(len(img_files), len(mask_files))), k=4)

fig, axes = plt.subplots(4, 3, figsize=(15, 20))
for i, idx in enumerate(samples):
    img  = np.array(Image.open(f'{IMG_DIR}/{img_files[idx]}'))
    mask = np.array(Image.open(f'{MASK_DIR}/{mask_files[idx]}')) / 255
    overlay = img.copy()
    overlay[mask > 0.5] = [255, 80, 0]   # Highlight roads in orange

    axes[i, 0].imshow(img);       axes[i, 0].set_title('Satellite Image')
    axes[i, 1].imshow(mask, cmap='gray'); axes[i, 1].set_title('Road Mask')
    axes[i, 2].imshow(overlay);   axes[i, 2].set_title('Overlay')
    for ax in axes[i]: ax.axis('off')

plt.suptitle('SpaceNet Road Detection — Sample Chips', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/sample_chips.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Saved to {DRIVE_BASE}/sample_chips.png')

In [ ]:
# ── Step 10: PyTorch Dataset ─────────────────────────────────────────────
import os, glob
from typing import Tuple

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2


def get_train_transforms(size=512):
    return A.Compose([
        A.RandomCrop(height=size, width=size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(0.3, 0.3, p=0.5),
        A.GaussianBlur((3, 7), p=0.3),
        A.GaussNoise(var_limit=(10, 50), p=0.3),
        # ── Occlusion simulation (clouds / monsoon) ──────────
        A.CoarseDropout(max_holes=8, max_height=64, max_width=64,
                        min_holes=2, fill_value=255, p=0.4),
        A.RandomShadow(p=0.3),
        # ── Normalise ────────────────────────────────────────
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])

def get_val_transforms(size=512):
    return A.Compose([
        A.CenterCrop(height=size, width=size),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])


class SpaceNetRoadDataset(Dataset):
    def __init__(self, img_dir: str, mask_dir: str, transform=None):
        self.img_dir   = img_dir
        self.mask_dir  = mask_dir
        self.transform = transform
        self.ids = [
            f for f in os.listdir(img_dir)
            if os.path.exists(os.path.join(mask_dir, f))
        ]
        print(f'Dataset: {len(self.ids)} samples')

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        fname = self.ids[idx]
        img   = np.array(Image.open(f'{self.img_dir}/{fname}').convert('RGB'))
        mask  = np.array(Image.open(f'{self.mask_dir}/{fname}').convert('L')) / 255.0
        mask  = (mask > 0.5).astype(np.float32)

        if self.transform:
            out  = self.transform(image=img, mask=mask)
            img  = out['image']   # [3, H, W] tensor
            mask = out['mask'].unsqueeze(0)  # [1, H, W]
        return img, mask


# ── Train / Val split ────────────────────────────────────────────────────
all_ids  = sorted(os.listdir(IMG_DIR))
n_train  = int(len(all_ids) * CFG['train_split'])
train_ids, val_ids = all_ids[:n_train], all_ids[n_train:]
print(f'Train: {len(train_ids)} | Val: {len(val_ids)}')

# Write split files
with open('/content/train.txt', 'w') as f: f.write('\n'.join(train_ids))
with open('/content/val.txt',   'w') as f: f.write('\n'.join(val_ids))

train_ds = SpaceNetRoadDataset(IMG_DIR, MASK_DIR, get_train_transforms(CFG['image_size']))
val_ds   = SpaceNetRoadDataset(IMG_DIR, MASK_DIR, get_val_transforms(CFG['image_size']))

train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                      num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                      num_workers=2, pin_memory=True)

print(f'Batches — Train: {len(train_dl)} | Val: {len(val_dl)}')

In [ ]:
# ── Step 11: Model + Loss + Optimizer ────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import SegformerForSemanticSegmentation


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        b = logits.shape[0]
        p = probs.view(b, -1)
        t = targets.view(b, -1)
        inter = (p * t).sum(dim=1)
        dice  = (2 * inter + self.smooth) / (p.sum(dim=1) + t.sum(dim=1) + self.smooth)
        return 1.0 - dice.mean()


class CombinedLoss(nn.Module):
    def __init__(self, dice_weight=0.5):
        super().__init__()
        self.dw  = dice_weight
        self.bw  = 1.0 - dice_weight
        self.dice = DiceLoss()
        self.bce  = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        return self.dw * self.dice(logits, targets) + self.bw * self.bce(logits, targets)


def compute_iou(logits, targets, threshold=0.5):
    probs = torch.sigmoid(logits) > threshold
    inter = (probs & targets.bool()).float().sum()
    union = (probs | targets.bool()).float().sum()
    return (inter / (union + 1e-6)).item()


# ── Model ────────────────────────────────────────────────────────────────
print(f'Loading SegFormer-B2 from HuggingFace...')
model = SegformerForSemanticSegmentation.from_pretrained(
    'nvidia/mit-b2',
    num_labels=1,
    ignore_mismatched_sizes=True,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

# ── Loss & Optimizer ─────────────────────────────────────────────────────
criterion = CombinedLoss(dice_weight=CFG['dice_weight'])

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG['lr'],
    weight_decay=CFG['weight_decay'],
)

# Cosine LR schedule with linear warmup
from torch.optim.lr_scheduler import OneCycleLR
scheduler = OneCycleLR(
    optimizer,
    max_lr=CFG['lr'],
    epochs=CFG['num_epochs'],
    steps_per_epoch=len(train_dl),
    pct_start=CFG['warmup_epochs'] / CFG['num_epochs'],
)

scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))
print('✅ Model, loss, optimizer, scheduler ready')

In [ ]:
# ── Step 12: Training Loop ────────────────────────────────────────────────
import time
from tqdm import tqdm

best_iou    = 0.0
train_losses, val_losses, val_ious = [], [], []


def train_epoch(model, loader, optimizer, criterion, scaler, scheduler):
    model.train()
    total_loss = 0.0
    for imgs, masks in tqdm(loader, desc='Train', leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=(device=='cuda')):
            outputs = model(pixel_values=imgs)
            logits  = outputs.logits  # [B, 1, H/4, W/4]
            logits  = F.interpolate(logits, size=imgs.shape[-2:],
                                    mode='bilinear', align_corners=False)
            loss    = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, total_iou = 0.0, 0.0
    for imgs, masks in tqdm(loader, desc='Val  ', leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        with torch.cuda.amp.autocast(enabled=(device=='cuda')):
            outputs = model(pixel_values=imgs)
            logits  = F.interpolate(outputs.logits, size=imgs.shape[-2:],
                                    mode='bilinear', align_corners=False)
            loss    = criterion(logits, masks)
        total_loss += loss.item()
        total_iou  += compute_iou(logits, masks)
    return total_loss / len(loader), total_iou / len(loader)


# ── Main loop ────────────────────────────────────────────────────────────
print(f'Starting training: {CFG["num_epochs"]} epochs | device={device}')

for epoch in range(1, CFG['num_epochs'] + 1):
    t0 = time.time()

    train_loss          = train_epoch(model, train_dl, optimizer, criterion, scaler, scheduler)
    val_loss, val_iou   = val_epoch(model, val_dl, criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_ious.append(val_iou)

    elapsed = time.time() - t0
    print(
        f'Epoch [{epoch:3d}/{CFG["num_epochs"]}] '
        f'Train: {train_loss:.4f} | Val: {val_loss:.4f} | '
        f'IoU: {val_iou:.4f} | LR: {scheduler.get_last_lr()[0]:.2e} | '
        f'Time: {elapsed:.1f}s'
    )

    # Save best
    if val_iou > best_iou:
        best_iou = val_iou
        save_path = f"{CFG['weights_dir']}/{CFG['best_model_name']}"
        torch.save(model.state_dict(), save_path)
        print(f'  ✅ Best model saved! IoU={best_iou:.4f} → {save_path}')

    # Periodic checkpoint
    if epoch % CFG['checkpoint_every'] == 0:
        ckpt_path = f"{CFG['weights_dir']}/checkpoint_epoch{epoch:03d}.pth"
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_iou': best_iou,
            'train_losses': train_losses,
            'val_ious': val_ious,
        }, ckpt_path)
        print(f'  📦 Checkpoint saved: {ckpt_path}')

print(f'\n🏁 Training complete! Best IoU: {best_iou:.4f}')

In [ ]:
# ── Step 13: Plot Training Curves ────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train Loss', color='#EF4444')
axes[0].plot(val_losses,   label='Val Loss',   color='#3B82F6')
axes[0].set_title('Loss Curves')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(val_ious, label='Validation IoU', color='#10B981')
axes[1].axhline(y=best_iou, color='#F59E0B', linestyle='--', label=f'Best IoU: {best_iou:.4f}')
axes[1].set_title('Validation IoU')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('IoU')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('SegFormer Road Detection — Training Metrics', fontsize=13)
plt.tight_layout()

plot_path = f'{DRIVE_BASE}/training_curves.png'
plt.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved to {plot_path}')

In [ ]:
# ── Step 14: Authenticate GEE & Fetch Coimbatore Imagery ─────────────────
import ee

# Authenticate (opens browser link once per session)
ee.Authenticate()
ee.Initialize(project='your-gee-project-id')  # replace with your project ID

# Define Coimbatore AOI
bbox = CFG['coimbatore_bbox']  # [xmin, ymin, xmax, ymax]
coimbatore_roi = ee.Geometry.Rectangle(bbox)

# Fetch cloud-masked Sentinel-2 composite
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_mask  = qa.bitwiseAnd(1 << 10).eq(0)
    cirrus_mask = qa.bitwiseAnd(1 << 11).eq(0)
    return image.updateMask(cloud_mask.And(cirrus_mask)).divide(10000)

collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(coimbatore_roi)
    .filterDate('2024-01-01', '2024-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_s2_clouds)
)

n = collection.size().getInfo()
print(f'Found {n} cloud-free Sentinel-2 scenes over Coimbatore')

composite = collection.median().select(['B4', 'B3', 'B2']).clip(coimbatore_roi)

# Thumbnail URL
thumb_url = composite.getThumbURL({
    'min': 0.0, 'max': 0.3,
    'region': coimbatore_roi,
    'dimensions': 1024,
    'format': 'png',
})
print(f'Thumbnail: {thumb_url}')

In [ ]:
# ── Step 15: Display Coimbatore Satellite Image ───────────────────────────
import requests
from PIL import Image
import matplotlib.pyplot as plt
import io

response = requests.get(thumb_url)
coimbatore_img = Image.open(io.BytesIO(response.content)).convert('RGB')

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(coimbatore_img)
ax.set_title(
    'Coimbatore, Tamil Nadu — Sentinel-2 True Colour (2024)\n'
    f'Bbox: {bbox} | {n} scenes composited',
    fontsize=12
)
ax.axis('off')
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/coimbatore_satellite.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Coimbatore satellite image displayed and saved')

In [ ]:
# ── Step 16: Export Coimbatore GeoTIFF to Drive ───────────────────────────
# Full-resolution 10m Sentinel-2 GeoTIFF for inference

composite_full = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(coimbatore_roi)
    .filterDate('2024-01-01', '2024-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_s2_clouds)
    .median()
    .select(['B4', 'B3', 'B2'])
    .clip(coimbatore_roi)
)

task = ee.batch.Export.image.toDrive(
    image=composite_full,
    description='coimbatore_sentinel2_2024',
    folder='reroute-ai/coimbatore_data',
    fileNamePrefix='coimbatore_s2_2024',
    region=coimbatore_roi,
    scale=10,               # 10m native Sentinel-2 resolution
    crs='EPSG:4326',
    maxPixels=1e13,
)
task.start()
print(f'GEE Export Task started!')
print(f'Task ID: {task.id}')
print('Check status at: https://code.earthengine.google.com/tasks')

In [ ]:
# ── Step 17: Run Inference on Coimbatore ─────────────────────────────────
# After GEE export completes, run the model on Coimbatore

import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Load best model
best_model_path = f"{CFG['weights_dir']}/{CFG['best_model_name']}"
from transformers import SegformerForSemanticSegmentation

inf_model = SegformerForSemanticSegmentation.from_pretrained(
    'nvidia/mit-b2', num_labels=1, ignore_mismatched_sizes=True
).to(device)
inf_model.load_state_dict(torch.load(best_model_path, map_location=device))
inf_model.eval()
print(f'✅ Model loaded from {best_model_path}')

# Inference transform
inf_transform = A.Compose([
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2(),
])

def predict_image(img_array: np.ndarray, tile_size: int = 512, overlap: int = 64):
    """Sliding-window inference for large images."""
    h, w = img_array.shape[:2]
    stride = tile_size - overlap
    canvas = np.zeros((h, w), dtype=np.float32)
    counts = np.zeros((h, w), dtype=np.float32)

    for y in range(0, h, stride):
        for x in range(0, w, stride):
            y_end, x_end = min(y + tile_size, h), min(x + tile_size, w)
            tile = img_array[y:y_end, x:x_end]
            transformed = inf_transform(image=tile)['image'].unsqueeze(0).to(device)

            with torch.no_grad(), torch.cuda.amp.autocast(enabled=(device=='cuda')):
                logits = inf_model(pixel_values=transformed).logits
                prob   = torch.sigmoid(
                    F.interpolate(logits, size=(y_end-y, x_end-x),
                                  mode='bilinear', align_corners=False)
                ).squeeze().cpu().numpy()

            canvas[y:y_end, x:x_end] += prob
            counts[y:y_end, x:x_end] += 1

    return canvas / np.maximum(counts, 1)

# Use the downloaded Coimbatore thumbnail for quick test
coimbatore_arr = np.array(coimbatore_img)
print(f'Running inference on Coimbatore image: {coimbatore_arr.shape}')

prob_map = predict_image(coimbatore_arr, tile_size=CFG['image_size'])
road_mask = (prob_map > 0.5).astype(np.uint8)

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
axes[0].imshow(coimbatore_arr); axes[0].set_title('Coimbatore — Satellite')
axes[1].imshow(prob_map, cmap='RdYlGn'); axes[1].set_title('Road Probability Map')

overlay = coimbatore_arr.copy()
overlay[road_mask == 1] = [255, 80, 0]
axes[2].imshow(overlay); axes[2].set_title('Extracted Roads (Orange)')

for ax in axes: ax.axis('off')
plt.suptitle('Coimbatore Road Extraction — SegFormer', fontsize=14)
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/coimbatore_road_extraction.png', dpi=120)
plt.show()
print('✅ Coimbatore road extraction complete!')

In [ ]:
# ── Step 18: Summary ─────────────────────────────────────────────────────
print('=' * 60)
print('  Route Resilience AI — Training Complete')
print('  Focus: Coimbatore, Tamil Nadu, India')
print('=' * 60)
print(f'  Best Validation IoU : {best_iou:.4f}')
print(f'  Epochs trained      : {len(train_losses)}')
print(f'  Best model saved to : {DRIVE_BASE}/weights/{CFG["best_model_name"]}')
print()
print('  Next steps:')
print('  1. Download best_model.pth from Drive → ai/weights/best_model.pth')
print('  2. Run: python backend/main.py')
print('  3. POST /api/roads/extract with Coimbatore satellite image')
print('  4. POST /api/graph/build → route emergency vehicles')
print('=' * 60)